In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import minimize
from scipy.linalg import expm
import time
import warnings
warnings.filterwarnings('ignore')
 
from pyscf import gto, scf, ao2mo, mcscf
from openfermion import (FermionOperator, normal_ordered, jordan_wigner,
                         get_sparse_operator, count_qubits)

## __KONFIGURASI GLOBAL & DEFINISI MOLEKUL__ ##

In [2]:
ACTIVE_MOLECULE = 'CH4'
ANSATZ_TYPE     = 'k-UpCCGSD'
K_LAYERS        = 2
USE_SCBK        = True

MOLECULES = {
    'H2': {
        'description'  : 'Molekul hidrogen H2, basis STO-3G',
        'atom'         : lambda R: f'H 0 0 0; H 0 0 {R}',
        'basis'        : 'sto-3g',
        'charge'       : 0,
        'spin'         : 0,
        'n_electrons'  : 2,
        'n_frozen_core': 0,
        'n_active_orbs': 2,
        'bond_lengths' : np.array([0.5, 0.6, 0.7, 0.74, 0.8,
                                    0.9, 1.0, 1.2, 1.5, 2.0]),
    },
    'H2O': {
        'description'  : 'Molekul air H2O, basis STO-3G, active space CAS(4e,4o)',
        'atom'         : None,
        'basis'        : 'sto-3g',
        'charge'       : 0,
        'spin'         : 0,
        'n_electrons'  : 10,
        'n_frozen_core': 3,
        'n_active_orbs': 4,
        'bond_lengths' : None,
    },
    'CH4': {
        'description'  : 'Molekul metan CH4, basis STO-3G, active space CAS(8e,8o)',
        'atom'         : None,   # diisi di bawah
        'basis'        : 'sto-3g',
        'charge'       : 0,
        'spin'         : 0,
        'n_electrons'  : 10,
        'n_frozen_core': 1,      # C 1s core di-freeze
        'n_active_orbs': 8,      # 8 orbital aktif (NOON: 0.02<NOON<1.98, CAS(8,8))
        'bond_lengths' : None,   # diisi di bawah
    },
}

# ─── H2O geometry (dipertahankan untuk referensi) ──────────────────────
def h2o_geometry(R_OH=0.96, theta_deg=104.45):
    theta = np.radians(theta_deg / 2)
    x = R_OH * np.sin(theta)
    z = R_OH * np.cos(theta)
    return (f'O  0.000  0.000  0.000;'
            f'H  0.000  {x:.6f}  {z:.6f};'
            f'H  0.000 {-x:.6f}  {z:.6f}')

# ─── CH4 geometry: tetrahedral, parameter = panjang ikatan C-H ─────────
def ch4_geometry(R_CH=1.085):
    # Simpul tetrahedron: (±1,±1,±1)/sqrt(3)*R_CH, tanda genap +
    # Menjamin sudut H-C-H = arccos(-1/3) = 109.47 deg
    d = R_CH / np.sqrt(3)
    return (
        f'C   0.000000  0.000000  0.000000;'
        f'H   {d:.6f}  {d:.6f}  {d:.6f};'
        f'H   {d:.6f} {-d:.6f} {-d:.6f};'
        f'H  {-d:.6f}  {d:.6f} {-d:.6f};'
        f'H  {-d:.6f} {-d:.6f}  {d:.6f}'
    )

MOLECULES['H2O']['atom']         = h2o_geometry
MOLECULES['H2O']['bond_lengths'] = np.array([
    0.70, 0.80, 0.90, 0.96, 1.09, 1.20, 1.40, 1.60, 1.80, 2.00
])

MOLECULES['CH4']['atom']         = ch4_geometry
MOLECULES['CH4']['bond_lengths'] = np.array([
    0.70, 0.80, 0.90, 1.00, 1.085, 1.10,
    1.20, 1.40, 1.60, 1.80, 2.00, 2.50, 3.00
])

mol_cfg = MOLECULES[ACTIVE_MOLECULE]

## __BUILD HAMILTONIAN__ ##

In [3]:
def build_one_electron(integral, n_orb):
    H = FermionOperator()
    for p in range(n_orb):
        for q in range(n_orb):
            if abs(integral[p, q]) < 1e-12:
                continue
            for sigma in range(2):
                i = 2 * p + sigma
                j = 2 * q + sigma
                H += FermionOperator(((i, 1), (j, 0)), integral[p, q])
    return H

def build_two_electron(eri_mo, n_orb):
    H = FermionOperator()
    for p in range(n_orb):
        for q in range(n_orb):
            for r in range(n_orb):
                for s in range(n_orb):
                    g = eri_mo[p, q, r, s]
                    if abs(g) < 1e-12:
                        continue
                    for sigma in range(2):
                        for tau in range(2):
                            i = 2 * p + sigma
                            j = 2 * r + tau
                            k = 2 * s + tau
                            l = 2 * q + sigma
                            H += FermionOperator(
                                ((i, 1), (j, 1), (k, 0), (l, 0)),
                                0.5 * g
                            )
    return H

In [4]:
def compute_scbk_isometry(H_jw_mat, H_scbk_mat, n_active_e, n_q_jw):
    '''
    Bangun isometri U: C^dim_scbk -> C^dim_jw
    dengan cara memilih eigenvector JW yang eigenvalue-nya cocok
    dengan eigenvalue SCBK (sektor N-elektron).
    '''
    evals_jw, evecs_jw = np.linalg.eigh(H_jw_mat)
    evals_sc, evecs_sc = np.linalg.eigh(H_scbk_mat)

    tol = 1e-5
    selected_cols = []
    for e_sc in evals_sc:
        diffs = np.abs(evals_jw - e_sc)
        idx   = int(np.argmin(diffs))
        selected_cols.append(idx)

    dim_jw   = H_jw_mat.shape[0]
    dim_scbk = H_scbk_mat.shape[0]
    U = evecs_jw[:, selected_cols]   # shape: (dim_jw, dim_scbk)
    assert U.shape == (dim_jw, dim_scbk), f"Shape U salah: {U.shape}"

    H_check = U.T.conj() @ H_jw_mat @ U
    err = np.max(np.abs(H_check - H_scbk_mat))
    if err > 1e-4:
        print(f"  [WARN] Isometri tidak sempurna: max|H_check - H_scbk| = {err:.2e}")
    else:
        print(f"  [OK] Isometri valid: max error = {err:.2e}")
    return U

In [5]:
def build_hamiltonian_v2(mol_cfg, bond_length=None, use_scbk=False):
    '''
    Returns: H_mat, T_mat, V_mat, E_core, n_q, n_orb, n_active_e,
             n_spin_orbs, n_q_jw, U_scbk, E_hf_mf, H_jw
    H_mat sudah termasuk E_core (energi repulsi inti / frozen core).
    '''
    mol = gto.Mole()
    atom_fn    = mol_cfg['atom']
    mol.atom   = atom_fn(bond_length) if bond_length is not None else atom_fn()
    mol.basis  = mol_cfg['basis']
    mol.charge = mol_cfg['charge']
    mol.spin   = mol_cfg['spin']
    mol.verbose = 0
    mol.build()

    mf = scf.RHF(mol)
    mf.run()
    E_hf_mf    = mf.e_tot
    n_frozen   = mol_cfg['n_frozen_core']
    n_active   = mol_cfg['n_active_orbs']
    n_elec_tot = mol_cfg['n_electrons']

    n_active_e = n_elec_tot - 2 * n_frozen
    if n_active_e <= 0:
        raise ValueError(f"n_active_e={n_active_e} tidak valid. "
                         f"Cek n_frozen_core={n_frozen} vs n_electrons={n_elec_tot}.")
    if n_active_e % 2 != mol_cfg.get('spin', 0) % 2:
        raise ValueError(f"n_active_e={n_active_e} tidak konsisten dengan spin.")

    if n_frozen > 0 or n_active is not None:
        n_act = n_active if n_active is not None else mol.nao_nr() - n_frozen
        if n_active_e > 2 * n_act:
            raise ValueError(f"Elektron aktif ({n_active_e}) melebihi kapasitas "
                             f"2*n_act={2*n_act}. Perbesar n_active_orbs.")

        cas   = mcscf.CASCI(mf, n_act, n_active_e)
        cas.verbose = 0
        h1, E_core = cas.get_h1eff()
        h2_cas     = cas.get_h2eff()
        h2         = ao2mo.restore(1, h2_cas, n_act).reshape(n_act, n_act, n_act, n_act)
        n_orb      = n_act

        T_ferm = normal_ordered(build_one_electron(h1, n_orb))
        V_ferm = normal_ordered(build_two_electron(h2, n_orb))
        H_active = T_ferm + V_ferm
        n_spin_orbs = 2 * n_orb
        H_ferm = H_active + FermionOperator((), E_core)
        T_ferm_nocore = T_ferm
        V_ferm_nocore = V_ferm
    else:
        C      = mf.mo_coeff
        n_orb  = C.shape[1]
        n_active_e = n_elec_tot
        E_core = mol.energy_nuc()
        T_ao   = mol.intor('int1e_kin')
        V_ao   = mol.intor('int1e_nuc')
        T_mo   = C.T @ T_ao @ C
        V_mo   = C.T @ V_ao @ C
        eri_mo = ao2mo.kernel(mol, C, compact=False).reshape(n_orb, n_orb, n_orb, n_orb)
        T_ferm = normal_ordered(build_one_electron(T_mo, n_orb) +
                                build_one_electron(V_mo, n_orb))
        V_ferm = normal_ordered(build_two_electron(eri_mo, n_orb))
        H_ferm = T_ferm + V_ferm + FermionOperator((), E_core)
        T_ferm_nocore = T_ferm
        V_ferm_nocore = V_ferm

    n_spin_orbs = 2 * n_orb
    H_q_jw = jordan_wigner(H_ferm)
    T_q_jw = jordan_wigner(T_ferm)
    V_q_jw = jordan_wigner(V_ferm)
    n_q_jw = count_qubits(H_q_jw)
    H_jw   = get_sparse_operator(H_q_jw, n_qubits=n_q_jw).toarray()
    T_jw   = get_sparse_operator(T_q_jw, n_qubits=n_q_jw).toarray()
    V_jw   = get_sparse_operator(V_q_jw, n_qubits=n_q_jw).toarray()

    if use_scbk:
        from openfermion.transforms import symmetry_conserving_bravyi_kitaev
        H_q_sc = symmetry_conserving_bravyi_kitaev(H_ferm, n_spin_orbs, n_active_e)
        Tq_sc  = symmetry_conserving_bravyi_kitaev(T_ferm_nocore, n_spin_orbs, n_active_e)
        Vq_sc  = symmetry_conserving_bravyi_kitaev(V_ferm_nocore, n_spin_orbs, n_active_e)
        n_q    = count_qubits(H_q_sc)
        H_mat  = get_sparse_operator(H_q_sc, n_qubits=n_q).toarray()
        T_mat  = get_sparse_operator(Tq_sc,  n_qubits=n_q).toarray()
        V_mat  = get_sparse_operator(Vq_sc,  n_qubits=n_q).toarray()
        U_scbk = compute_scbk_isometry(H_jw, H_mat, n_active_e, n_q_jw)
    else:
        n_q    = n_q_jw
        H_mat  = H_jw
        T_mat  = T_jw
        V_mat  = V_jw
        U_scbk = None

    return (H_mat, T_mat, V_mat, E_core, n_q, n_orb, n_active_e,
            n_spin_orbs, n_q_jw, U_scbk, E_hf_mf, H_jw)

In [6]:
def verify_energy(H_mat, E_core, cas_ref_energy=None):
    evals = np.linalg.eigvalsh(H_mat)
    E_gs  = evals[0].real
    print(f"  Ground state energy (dari H_mat)  : {E_gs:.8f} Ha")
    print(f"  E_core                             : {E_core:.8f} Ha")
    print(f"  E_active (E_gs - E_core)           : {E_gs - E_core:.8f} Ha")
    if cas_ref_energy is not None:
        print(f"  Referensi CASCI                    : {cas_ref_energy:.8f} Ha")
        print(f"  Selisih                            : {abs(E_gs - cas_ref_energy):.2e} Ha")
    return E_gs

In [7]:
def build_hamiltonian(bond_length):
    H, T, V, E_nuc, n_q, n_orb, _, _, _, _, _, _ = build_hamiltonian_v2(
        MOLECULES['CH4'], bond_length=bond_length, use_scbk=USE_SCBK)
    return H, T, V, E_nuc, n_q, n_orb

## __QUANTUM CIRCUITS__ ##

In [8]:
def ry_gate(theta):
    c, s = np.cos(theta / 2), np.sin(theta / 2)
    return np.array([[c, -s], [s, c]], dtype=complex)

In [9]:
def kron_gate(gate_2x2, qubit, n_qubits):
    ops = [np.eye(2, dtype=complex)] * n_qubits
    ops[qubit] = gate_2x2
    result = ops[0]
    for op in ops[1:]:
        result = np.kron(result, op)
    return result

In [10]:
def cnot_full(n_qubits, control, target):
    dim = 2 ** n_qubits
    mat = np.zeros((dim, dim), dtype=complex)
    for j in range(dim):
        ctrl_bit = (j >> (n_qubits - 1 - control)) & 1
        if ctrl_bit:
            tgt_pos = n_qubits - 1 - target
            i = j ^ (1 << tgt_pos)
        else:
            i = j
        mat[i, j] = 1.0
    return mat

In [11]:
def build_linear_entangling_layer(n_qubits):
    mat = np.eye(2 ** n_qubits, dtype=complex)
    for q in range(n_qubits - 1):
        mat = cnot_full(n_qubits, q, q + 1) @ mat
    return mat

## __STATE PREPARATION: HARTREE-FOCK__ ##

In [12]:
def prepare_hf_state(n_qubits, n_electrons,
                     use_scbk=False, U_scbk=None, n_q_jw=None, H_mat=None, **kwargs):
    '''
    Params
    ------
    n_qubits    : jumlah qubit (setelah encoding, bisa < n_electrons jika SCBK)
    n_electrons : jumlah elektron aktif
    use_scbk    : True -> proyeksikan HF state ke ruang SCBK
    U_scbk      : isometri SCBK (dim_jw x dim_scbk)
    '''
    if use_scbk and H_mat is not None:
        dim = 2 ** n_qubits
        diag_energies = np.array([H_mat[i, i].real for i in range(dim)])
        hf_idx = int(np.argmin(diag_energies))
    else:
        hf_idx = sum(1 << (n_qubits - 1 - i) for i in range(n_electrons))

    psi = np.zeros(2 ** n_qubits, dtype=complex)
    psi[hf_idx] = 1.0
    return psi

## __ANSATZ__ ##

### __1. UCCSD__ ###

In [13]:
def build_single_excitation_generator(i, a, n_qubits,
                                       use_scbk=False, U_scbk=None, n_q_jw=None,
                                       **kwargs):
    op       = FermionOperator(((a, 1), (i, 0))) - FermionOperator(((i, 1), (a, 0)))
    qubit_op = jordan_wigner(op)
    n_q_build = n_q_jw if (use_scbk and n_q_jw is not None) else n_qubits
    G_jw = get_sparse_operator(qubit_op, n_qubits=n_q_build).toarray()
    if use_scbk and U_scbk is not None:
        return U_scbk.T.conj() @ G_jw @ U_scbk
    return G_jw

In [14]:
def build_double_excitation_generator(i, j, a, b, n_qubits,
                                       use_scbk=False, U_scbk=None, n_q_jw=None,
                                       **kwargs):
    op       = (FermionOperator(((a, 1), (b, 1), (j, 0), (i, 0))) -
                FermionOperator(((i, 1), (j, 1), (b, 0), (a, 0))))
    qubit_op = jordan_wigner(op)
    n_q_build = n_q_jw if (use_scbk and n_q_jw is not None) else n_qubits
    G_jw = get_sparse_operator(qubit_op, n_qubits=n_q_build).toarray()
    if use_scbk and U_scbk is not None:
        return U_scbk.T.conj() @ G_jw @ U_scbk
    return G_jw

In [15]:
def get_uccsd_excitations(n_electrons, n_qubits):
    '''
    Enumerate semua pasangan (i->a) single dan (ij->ab) double excitation.
    Spin orbital: 0..n_electrons-1 = occupied, n_electrons..n_qubits-1 = virtual.
    '''
    n_occ   = n_electrons
    singles = [(i, a)
               for i in range(n_occ)
               for a in range(n_occ, n_qubits)]
    doubles = [(i, j, a, b)
               for i in range(n_occ)
               for j in range(i + 1, n_occ)
               for a in range(n_occ, n_qubits)
               for b in range(a + 1, n_qubits)]
    return singles, doubles

In [16]:
def count_uccsd_params(n_electrons, n_qubits):
    singles, doubles = get_uccsd_excitations(n_electrons, n_qubits)
    return len(singles) + len(doubles)

In [17]:
def prepare_ansatz_uccsd(params, n_qubits, n_electrons,
                          use_scbk=False, U_scbk=None, n_q_jw=None, H_mat=None, **kwargs):
    '''
    UCCSD ansatz (Trotterisasi):
        |psi(theta)> = [prod_k e^(theta_k G_k)] |HF>
    '''
    sc  = dict(use_scbk=use_scbk, U_scbk=U_scbk, n_q_jw=n_q_jw, H_mat=H_mat)
    psi = prepare_hf_state(n_qubits, n_electrons, **sc)
    singles, doubles = get_uccsd_excitations(n_electrons, n_qubits)
    idx = 0
    for (i, a) in singles:
        G   = build_single_excitation_generator(i, a, n_qubits, **sc)
        psi = expm(params[idx] * G) @ psi
        idx += 1
    for (i, j, a, b) in doubles:
        G   = build_double_excitation_generator(i, j, a, b, n_qubits, **sc)
        psi = expm(params[idx] * G) @ psi
        idx += 1
    return psi

In [18]:
prepare_ansatz = prepare_ansatz_uccsd

### __2. K-UpCCGSD__ ###

In [19]:
def get_kupccgsd_excitations(n_orb, n_qubits):
    # Generalized singles: semua pasangan spin-orbital (p,q), p<q
    gen_singles = [(p, q)
                   for p in range(n_qubits)
                   for q in range(p + 1, n_qubits)]
    # Paired doubles: orbital spasial p->q, keduanya doubly occupied
    # Spin-orbital: p_alpha=2p, p_beta=2p+1, q_alpha=2q, q_beta=2q+1
    paired_doubles = [(2 * p, 2 * p + 1, 2 * q, 2 * q + 1)
                      for p in range(n_orb)
                      for q in range(p + 1, n_orb)]
    return gen_singles, paired_doubles

In [20]:
def count_kupccgsd_params(n_orb, n_qubits, k=1):
    gs, pd_ = get_kupccgsd_excitations(n_orb, n_qubits)
    return k * (len(gs) + len(pd_))

In [21]:
def build_generators_kupccgsd(n_qubits, n_electrons, n_orb, k=1,
                               use_scbk=False, U_scbk=None, n_q_jw=None, H_mat=None, **kwargs):
    # Pre-compute semua generator sekali saja, simpan dalam list.
    sc = dict(use_scbk=use_scbk, U_scbk=U_scbk, n_q_jw=n_q_jw, H_mat=H_mat)
    gen_singles, paired_doubles = get_kupccgsd_excitations(n_orb, n_qubits)

    generators = []
    for (p, q) in gen_singles:
        G = build_single_excitation_generator(p, q, n_qubits, **sc)
        generators.append(G)
    for (i, j, a, b) in paired_doubles:
        G = build_double_excitation_generator(i, j, a, b, n_qubits, **sc)
        generators.append(G)
    return generators

def prepare_ansatz_kupccgsd_cached(params, n_qubits, n_electrons, n_orb=None, k=1,
                                    generators=None,
                                    use_scbk=False, U_scbk=None, n_q_jw=None, H_mat=None, **kwargs):
    # Versi cached: generator tidak dibangun ulang tiap evaluasi.
    if n_orb is None:
        n_orb = n_qubits // 2
    sc  = dict(use_scbk=use_scbk, U_scbk=U_scbk, n_q_jw=n_q_jw, H_mat=H_mat)
    psi = prepare_hf_state(n_qubits, n_electrons, **sc)

    n_per_layer = len(generators)
    for layer in range(k):
        idx = layer * n_per_layer
        for G in generators:
            psi = expm(params[idx] * G) @ psi
            idx += 1
    return psi

In [22]:
def prepare_ansatz_kupccgsd(params, n_qubits, n_electrons, n_orb=None, k=1,
                             use_scbk=False, U_scbk=None, n_q_jw=None, H_mat=None, **kwargs):
    # Mode on-the-fly: tidak cache generator (hemat memori, lebih lambat).
    if n_orb is None:
        n_orb = n_qubits // 2
    sc  = dict(use_scbk=use_scbk, U_scbk=U_scbk, n_q_jw=n_q_jw, H_mat=H_mat)
    psi = prepare_hf_state(n_qubits, n_electrons, **sc)
    gen_singles, paired_doubles = get_kupccgsd_excitations(n_orb, n_qubits)
    n_per_layer = len(gen_singles) + len(paired_doubles)
    for layer in range(k):
        idx = layer * n_per_layer
        for (p, q) in gen_singles:
            G   = build_single_excitation_generator(p, q, n_qubits, **sc)
            psi = expm(params[idx] * G) @ psi
            idx += 1
        for (i, j, a, b) in paired_doubles:
            G   = build_double_excitation_generator(i, j, a, b, n_qubits, **sc)
            psi = expm(params[idx] * G) @ psi
            idx += 1
    return psi

## __VARIATIONAL QUANTUM EIGENSOLVER__ ##

In [23]:
def expect_value(psi, H_mat):
    return float(np.real(psi.conj() @ H_mat @ psi))

In [24]:
def optimizer_cl(params, H_mat, n_qubits, n_electrons, history, ansatz_fn, ansatz_kwargs):
    psi = ansatz_fn(params, n_qubits, n_electrons, **ansatz_kwargs)
    E   = expect_value(psi, H_mat)
    history.append(E)
    return E

In [25]:
def run_vqe_v2(H_mat, n_qubits, n_electrons, n_orb=None,
               ansatz_type='k-UpCCGSD', k=1,
               use_scbk=False, U_scbk=None, n_q_jw=None,
               n_spin_orbs=None, E_core=0.0, E_hf_ref=None,
               method='COBYLA', n_restarts=5, seed=42, verbose=False):
    '''
    Jalankan VQE: minimasi E(theta) = <psi(theta)|H|psi(theta)> dengan optimizer klasik.

    Catatan untuk CH4 CAS(8,8) / 14 qubit SCBK:
      Ukuran Hilbert space: 2^14 = 16384
      Memori generator per layer: ~510 GB -> mode ON-THE-FLY (tidak di-cache).
      Gunakan method='L-BFGS-B' untuk konvergensi terbaik.
    '''
    if n_orb is None:
        n_orb = n_qubits // 2
    rng = np.random.RandomState(seed)

    ansatz_base_kwargs = dict(
        use_scbk=use_scbk, U_scbk=U_scbk, n_q_jw=n_q_jw, H_mat=H_mat,
    )

    if ansatz_type == 'UCCSD':
        ansatz_fn     = prepare_ansatz_uccsd
        n_params      = count_uccsd_params(n_electrons,
                            n_q_jw if (use_scbk and n_q_jw) else n_qubits)
        ansatz_kwargs = ansatz_base_kwargs
    else:   # k-UpCCGSD
        n_params = count_kupccgsd_params(n_orb, n_qubits, k=k)

        # ─── Auto memory check untuk generator caching ────────────────
        _gs, _pd  = get_kupccgsd_excitations(n_orb, n_qubits)
        n_gen     = len(_gs) + len(_pd)
        est_gb    = n_gen * (2**n_qubits)**2 * 16 / 1e9  # complex128

        if est_gb < 32.0:   # Cache jika estimasi < 32 GB (tuning sesuai node)
            print(f"  Pre-computing {n_gen} generators ({est_gb:.3f} GB)...", end=' ')
            t0 = time.time()
            generators_cache = build_generators_kupccgsd(
                n_qubits, n_electrons, n_orb, k=1, **ansatz_base_kwargs
            )
            print(f"done ({time.time()-t0:.2f}s)")
            ansatz_fn     = prepare_ansatz_kupccgsd_cached
            ansatz_kwargs = {**ansatz_base_kwargs, 'n_orb': n_orb, 'k': k,
                             'generators': generators_cache}
        else:
            # On-the-fly mode: bangun generator tiap evaluasi, hemat memori
            print(f"  Generator cache: {est_gb:.1f} GB > 32 GB -> mode on-the-fly.")
            ansatz_fn     = prepare_ansatz_kupccgsd
            ansatz_kwargs = {**ansatz_base_kwargs, 'n_orb': n_orb, 'k': k}

    best = {'E_vqe': np.inf, 'energy_history': [], 'n_iters': 0}

    opts_map = {
        'COBYLA'  : {'maxiter': 10000, 'rhobeg': 0.5},
        'BFGS'    : {'maxiter': 1000,  'gtol'  : 1e-8},
        'L-BFGS-B': {'maxiter': 500,   'ftol'  : 1e-9},
    }
    opts = opts_map.get(method, {'maxiter': 2000})

    for trial in range(n_restarts):
        history = []
        theta0  = (rng.uniform(-0.1, 0.1, n_params) if trial < max(n_restarts // 2, 1)
                   else rng.uniform(-np.pi, np.pi, n_params))

        res = minimize(
            optimizer_cl,
            theta0,
            args=(H_mat, n_qubits, n_electrons, history, ansatz_fn, ansatz_kwargs),
            method=method,
            options=opts,
        )

        if verbose:
            print(f"  Restart {trial+1}/{n_restarts}: "
                  f"E = {res.fun:.6f} Ha | {len(history):4d} iter | "
                  f"success={res.success}")

        if res.fun < best['E_vqe']:
            best = {
                'E_vqe'         : res.fun,
                'optimal_params': res.x.copy(),
                'energy_history': history.copy(),
                'n_iters'       : len(history),
                'trial'         : trial + 1,
                'success'       : res.success,
            }

    best['ansatz_type'] = ansatz_type
    best['n_params']    = n_params
    return best

## __WARM START: MP2__ ##

In [26]:
def get_mp2_initial_params(mol_cfg, bond_length, n_electrons, n_qubits):
    '''
    Warm start: gunakan amplitudo MP2 sebagai inisialisasi theta_0.
    Hanya untuk UCCSD ansatz.
    '''
    mol = gto.Mole()
    mol.atom   = mol_cfg['atom'](bond_length)
    mol.basis  = mol_cfg['basis']
    mol.charge = mol_cfg['charge']
    mol.spin   = mol_cfg['spin']
    mol.verbose = 0
    mol.build()

    mf = scf.RHF(mol).run()
    from pyscf import mp
    mp2 = mp.MP2(mf).run()
    t2 = mp2.t2   # shape: (nocc, nocc, nvir, nvir)
    nocc = t2.shape[0]
    nvir = t2.shape[2]
    doubles_flat = t2.reshape(-1)
    singles_init = np.zeros(nocc * nvir)
    params0 = np.concatenate([singles_init, doubles_flat])
    n_params = count_uccsd_params(n_electrons, n_qubits)
    if len(params0) >= n_params:
        return params0[:n_params]
    return np.pad(params0, (0, n_params - len(params0)))

## __VERIFIKASI KONFIGURASI__ ##

In [27]:
mol_cfg = MOLECULES[ACTIVE_MOLECULE]
print("=" * 70)
print(f"  VQE Upgrade - {ACTIVE_MOLECULE}")
print(f"  {mol_cfg['description']}")
print(f"  Ansatz  : {ANSATZ_TYPE}  (k={K_LAYERS})")
print(f"  Encoding: {'SCBK (-2 qubit)' if USE_SCBK else 'Jordan-Wigner'}")
print("=" * 70)

n_orb_info = mol_cfg['n_active_orbs']
n_q_info   = n_orb_info * 2
n_active_e = mol_cfg['n_electrons'] - 2 * mol_cfg['n_frozen_core']

if USE_SCBK:
    n_q_info -= 2

if ANSATZ_TYPE == 'UCCSD':
    n_p = count_uccsd_params(n_active_e, n_q_info)
else:
    n_p = count_kupccgsd_params(n_orb_info, n_q_info, k=K_LAYERS)

# Estimasi memori generator cache
_gs_info, _pd_info  = get_kupccgsd_excitations(n_orb_info, n_q_info)
n_gen_info          = len(_gs_info) + len(_pd_info)
est_gen_gb          = n_gen_info * (2**n_q_info)**2 * 16 / 1e9

print(f"\n  Perkiraan sumber daya:")
print(f"    Orbital aktif   : {n_orb_info}")
print(f"    Elektron aktif  : {n_active_e}")
print(f"    Qubit           : {n_q_info}")
print(f"    Dim Hilbert     : {2**n_q_info} x {2**n_q_info}")
print(f"    Jumlah param    : {n_p}")
print(f"\n  Estimasi memori generator ({n_gen_info} generator per layer):")
print(f"    Total           : ~{est_gen_gb:.1f} GB (complex128)")
cache_mode = 'CACHE (fast eval)' if est_gen_gb < 32 else 'ON-THE-FLY (hemat memori)'
print(f"    Mode            : {cache_mode}")

if ANSATZ_TYPE == 'k-UpCCGSD':
    n_p_uccsd = count_uccsd_params(n_active_e, n_q_info)
    print(f"\n  Perbandingan parameter ansatz:")
    print(f"    UCCSD          : {n_p_uccsd} params")
    print(f"    k-UpCCGSD k={K_LAYERS} : {n_p} params  ({n_p_uccsd/max(n_p,1):.1f}x lebih sedikit)")
    print(f"    Gen singles    : {len(_gs_info)}  |  Paired doubles: {len(_pd_info)}")

print()
# Estimasi waktu komputasi per titik PES (rough estimate)
n_params_est = n_p
print(f"  Estimasi kasar waktu per titik PES (L-BFGS-B):")
print(f"    Asumsi ~10s per evaluasi ansatz (on-the-fly, 14 qubit)")
print(f"    1 iter L-BFGS-B (grad): {(n_params_est+1)*10:.0f} s")
print(f"    500 iter x 3 restart  : {500*(n_params_est+1)*10*3/3600:.1f} jam")
print(f"    => Checkpoint SLURM sangat dibutuhkan!")

# Konfigurasi Global (backward-compat)
N_QUBITS    = 4
N_ELECTRONS = 2
N_PARAMS    = count_uccsd_params(N_ELECTRONS, N_QUBITS)
print(f"\nKonfigurasi UCCSD referensi (H2, 4 qubit):")
print(f"  n_params = {N_PARAMS}")

  VQE Upgrade - CH4
  Molekul metan CH4, basis STO-3G, active space CAS(8e,8o)
  Ansatz  : k-UpCCGSD  (k=2)
  Encoding: SCBK (-2 qubit)

  Perkiraan sumber daya:
    Orbital aktif   : 8
    Elektron aktif  : 8
    Qubit           : 14
    Dim Hilbert     : 16384 x 16384
    Jumlah param    : 238

  Estimasi memori generator (119 generator per layer):
    Total           : ~511.1 GB (complex128)
    Mode            : ON-THE-FLY (hemat memori)

  Perbandingan parameter ansatz:
    UCCSD          : 468 params
    k-UpCCGSD k=2 : 238 params  (2.0x lebih sedikit)
    Gen singles    : 91  |  Paired doubles: 28

  Estimasi kasar waktu per titik PES (L-BFGS-B):
    Asumsi ~10s per evaluasi ansatz (on-the-fly, 14 qubit)
    1 iter L-BFGS-B (grad): 2390 s
    500 iter x 3 restart  : 995.8 jam
    => Checkpoint SLURM sangat dibutuhkan!

Konfigurasi UCCSD referensi (H2, 4 qubit):
  n_params = 5


## __VQE DEMO PADA GEOMETRI EKUILIBRIUM CH4__ ##

In [ ]:
R_DEMO = 1.085   # Angstrom - panjang ikatan C-H ekuilibrium CH4 (STO-3G)
print(f"\n  VQE Demo: CH4, R_CH = {R_DEMO} Angstrom, Basis STO-3G")
print(f"  Active space: CAS({MOLECULES['CH4']['n_electrons'] - 2*MOLECULES['CH4']['n_frozen_core']}e,"
      f"{MOLECULES['CH4']['n_active_orbs']}o), frozen core = {MOLECULES['CH4']['n_frozen_core']}")


  VQE Demo: CH4, R_CH = 1.085 Angstrom, Basis STO-3G
  Active space: CAS(8e,8o), frozen core = 1


: 

In [29]:
# Build Hamiltonian CH4 pada ekuilibrium
H_demo, T_demo, V_demo, E_core_d, n_q_d, n_orb_d, n_ae_d, \
    n_spin_orbs_d, n_q_jw_d, U_scbk_d, E_hf_mf_d, H_jw_ref = \
    build_hamiltonian_v2(mol_cfg, bond_length=R_DEMO, use_scbk=USE_SCBK)

sc_kw = dict(use_scbk=USE_SCBK, U_scbk=U_scbk_d, n_q_jw=n_q_jw_d)

: 

: 

In [ ]:
# Energi eksak (FCI diagonalisasi penuh Hilbert space)
evals_d, evecs_d = np.linalg.eigh(H_demo)
E_exact_d = float(evals_d[0].real)
print(f"  Dimensi Hilbert space: {H_demo.shape}")
print(f"  E_exact (FCI)        : {E_exact_d:.8f} Ha")

In [ ]:
# Energi Hartree-Fock pada theta = 0
psi_hf_d = prepare_hf_state(n_q_d, n_ae_d, H_mat=H_demo, **sc_kw)
E_hf_d   = expect_value(psi_hf_d, H_demo)

print(f"E_HF dari ansatz (theta=0) : {E_hf_d:.8f}")
print(f"E_HF referensi (PySCF)     : {E_hf_mf_d:.8f}")
print(f"Selisih                    : {abs(E_hf_d - E_hf_mf_d):.2e}")

In [ ]:
# Analisis HF state CH4
psi_test = prepare_hf_state(n_q_d, n_ae_d)
hf_idx   = np.argmax(np.abs(psi_test))
print(f"n_qubits      : {n_q_d}")
print(f"n_electrons   : {n_ae_d}")
print(f"HF index      : {hf_idx}")
print(f"HF bitstring  : |{hf_idx:0{n_q_d}b}>")
print(f"E_HF (diag)   : {H_demo[hf_idx, hf_idx].real:.8f}")
evals_check, _ = np.linalg.eigh(H_demo)
print(f"E_FCI         : {evals_check[0].real:.8f}")
print(f"Selisih HF-FCI: {abs(H_demo[hf_idx,hf_idx].real - evals_check[0].real):.4f} Ha")

In [ ]:
print(f"\n  Qubit aktual   : {n_q_d}")
print(f"  Dim Hilbert    : {2**n_q_d} x {2**n_q_d}")
print(f"  E_nuc/E_core   : {E_core_d:.6f} Ha")
print(f"  E_exact (FCI)  : {E_exact_d:.8f} Ha")
print(f"  E_HF           : {E_hf_d:.8f} Ha  |  err = {abs(E_hf_d-E_exact_d):.2e} Ha")

In [ ]:
n_params_vqe = count_kupccgsd_params(n_orb_d, n_q_d, k=K_LAYERS)
print(f"n_params (k-UpCCGSD k={K_LAYERS}): {n_params_vqe}")

In [ ]:
# Benchmark 1 evaluasi ansatz CH4
sc_kw_bench = dict(use_scbk=USE_SCBK, U_scbk=U_scbk_d, n_q_jw=n_q_jw_d, H_mat=H_demo)
params_test  = np.zeros(count_kupccgsd_params(n_orb_d, n_q_d, k=K_LAYERS))

# Estimasi memori untuk memilih mode
_gs_b, _pd_b = get_kupccgsd_excitations(n_orb_d, n_q_d)
n_gen_b      = len(_gs_b) + len(_pd_b)
est_gb_b     = n_gen_b * (2**n_q_d)**2 * 16 / 1e9
print(f"n_generators per layer  : {n_gen_b}")
print(f"Est. cache memory       : {est_gb_b:.2f} GB")
print(f"Mode                    : {'CACHE' if est_gb_b < 32 else 'ON-THE-FLY'}")
print()

# Benchmark (hanya 2 iterasi karena CH4 besar)
N_BENCH = 2
t0 = time.time()
for _ in range(N_BENCH):
    prepare_ansatz_kupccgsd(params_test, n_q_d, n_ae_d,
                             n_orb=n_orb_d, k=K_LAYERS, **sc_kw_bench)
t_eval = (time.time()-t0)/N_BENCH

n_p = len(params_test)
print(f"Waktu 1 evaluasi ansatz   : {t_eval:.2f} s")
print(f"1 iter L-BFGS-B (+ grad) : {(n_p+1)*t_eval:.1f} s")
print(f"Est. 500 iter x 3 restart: {500*(n_p+1)*t_eval*3/3600:.1f} jam")
print(f"=> Sangat perlu SLURM checkpoint!")

In [ ]:
# Jalankan VQE demo pada geometri ekuilibrium CH4
print(f"Menjalankan VQE k-UpCCGSD k={K_LAYERS} untuk CH4 di R={R_DEMO} A ...")
vqe_d = run_vqe_v2(
    H_demo, n_q_d, n_ae_d,
    n_orb       = n_orb_d,
    ansatz_type = ANSATZ_TYPE,
    k           = K_LAYERS,
    use_scbk    = USE_SCBK,
    U_scbk      = U_scbk_d,
    n_q_jw      = n_q_jw_d,
    E_core      = E_core_d,
    method      = 'L-BFGS-B',
    n_restarts  = 3,
    seed        = 42,
    verbose     = True)

In [ ]:
E_vqe_d  = vqe_d['E_vqe']
err_vqe  = abs(E_vqe_d - E_exact_d)
corr_pct = ((E_hf_d - E_vqe_d) / (E_hf_d - E_exact_d) * 100
            if abs(E_hf_d - E_exact_d) > 1e-10 else 0.0)

In [ ]:
print(f"\n  -- Hasil VQE CH4 Demo ({R_DEMO} A) ----------------------------")
print(f"    E_VQE   = {E_vqe_d:.8f} Ha")
print(f"    E_exact = {E_exact_d:.8f} Ha")
acc_tag = 'Chem. Acc.' if err_vqe < 1.6e-3 else 'di luar chem. acc.'
print(f"    |Delta_E| = {err_vqe:.2e} Ha  ({acc_tag})")
print(f"    Korelasi tertangkap : {corr_pct:.1f}%")
print(f"  -----------------------------------------------------------")

## __CHECKPOINT SETUP — SLURM COMPUTE NODE__ ##

Cell ini menyiapkan sistem checkpoint untuk loop PES yang berat.  
Fitur:
- **Atomic save** setelah setiap titik PES (via tmp + rename POSIX)
- **Deteksi SIGTERM / SIGUSR1 / SIGUSR2** dari SLURM scheduler
- **Resume otomatis** jika job di-restart (skip titik yang sudah selesai)
- **Log file** terpisah untuk tracking per-titik  

Tips di SBATCH script:
```
#SBATCH --signal=USR1@120   # SIGUSR1 dikirim 120 detik sebelum batas waktu
#SBATCH --signal=TERM@60    # SIGTERM dikirim 60 detik sebelum batas waktu
```

In [ ]:
import pickle
import os
import signal
import pathlib
from datetime import datetime, timedelta

# ─── Konfigurasi path checkpoint ──────────────────────────────────────
CKPT_DIR  = pathlib.Path("checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

_cfg_tag      = f"CH4_{ANSATZ_TYPE}_k{K_LAYERS}_scbk{USE_SCBK}"
CKPT_FILE     = CKPT_DIR / f"pes_{_cfg_tag}.pkl"
CKPT_LOG_FILE = CKPT_DIR / f"pes_{_cfg_tag}.log"

# ─── Global state ─────────────────────────────────────────────────────
_SLURM_TERMINATE = False
_T_JOB_START     = time.time()

# ─── Logging ke file ──────────────────────────────────────────────────
def _ckpt_log(msg: str):
    ts   = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f"[{ts}] {msg}"
    with open(CKPT_LOG_FILE, 'a') as fh:
        fh.write(line + '\n')

# ─── Signal handler: SIGTERM / SIGUSR1 dari SLURM ─────────────────────
def _slurm_signal_handler(signum, frame):
    global _SLURM_TERMINATE
    _SLURM_TERMINATE = True
    sig_name = signal.Signals(signum).name
    elapsed  = timedelta(seconds=int(time.time() - _T_JOB_START))
    msg = (f"SIGNAL {sig_name} diterima! "
           f"Elapsed: {elapsed}. Checkpoint akan disimpan, keluar...")
    print(f"\n  [SLURM] {msg}")
    _ckpt_log(f"[SIGNAL] {msg}")

for _sig in (signal.SIGTERM, signal.SIGUSR1, signal.SIGUSR2):
    try:
        signal.signal(_sig, _slurm_signal_handler)
    except (OSError, AttributeError, ValueError):
        pass   # Tidak semua sinyal tersedia di semua platform

# ─── Simpan checkpoint (atomic: tulis .tmp lalu rename) ───────────────
def save_checkpoint(results: list, done_set: set):
    ckpt = {
        'version'    : 1,
        'molecule'   : 'CH4',
        'ansatz_type': ANSATZ_TYPE,
        'k_layers'   : K_LAYERS,
        'use_scbk'   : USE_SCBK,
        'timestamp'  : datetime.now().isoformat(),
        'results'    : results,
        'done_set'   : done_set,
    }
    tmp_path = str(CKPT_FILE) + '.tmp'
    with open(tmp_path, 'wb') as fh:
        pickle.dump(ckpt, fh, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp_path, CKPT_FILE)   # atomic rename (POSIX)
    msg = f"Saved {len(done_set)} PES points -> {CKPT_FILE}"
    print(f"  [CKPT] {msg}")
    _ckpt_log(msg)

# ─── Load checkpoint ──────────────────────────────────────────────────
def load_checkpoint():
    if not CKPT_FILE.exists():
        print("  [CKPT] Tidak ada checkpoint sebelumnya. Mulai dari awal.")
        _ckpt_log("Fresh start - no checkpoint found.")
        return [], set()

    with open(CKPT_FILE, 'rb') as fh:
        ckpt = pickle.load(fh)

    results  = ckpt.get('results', [])
    done_set = ckpt.get('done_set', set())
    ts       = ckpt.get('timestamp', 'N/A')

    # Validasi konfigurasi konsisten
    cfg_ok = (ckpt.get('molecule')    == 'CH4'       and
              ckpt.get('ansatz_type') == ANSATZ_TYPE  and
              ckpt.get('k_layers')    == K_LAYERS     and
              ckpt.get('use_scbk')    == USE_SCBK)
    if not cfg_ok:
        print("  [CKPT] PERINGATAN: konfigurasi checkpoint berbeda dengan sesi ini!")
        print(f"         Checkpoint: mol={ckpt.get('molecule')}, "
              f"ansatz={ckpt.get('ansatz_type')}, k={ckpt.get('k_layers')}, "
              f"scbk={ckpt.get('use_scbk')}")
        print(f"         Sekarang  : mol=CH4, ansatz={ANSATZ_TYPE}, "
              f"k={K_LAYERS}, scbk={USE_SCBK}")
        print("  [CKPT] Hapus CKPT_FILE secara manual jika ingin mulai dari awal.")

    print(f"  [CKPT] Checkpoint ditemukan  : {CKPT_FILE}")
    print(f"  [CKPT] Timestamp             : {ts}")
    print(f"  [CKPT] Titik selesai         : {len(done_set)}")
    print(f"  [CKPT] R values selesai (A)  : {sorted(done_set)}")
    _ckpt_log(f"Loaded checkpoint: {len(done_set)} points (ts={ts})")
    return results, done_set

# ─── Status ───────────────────────────────────────────────────────────
_slurm_job_id = os.environ.get('SLURM_JOB_ID', 'N/A (bukan SLURM / Jupyter)')
print(f"  [CKPT] Checkpoint file : {CKPT_FILE}")
print(f"  [CKPT] Log file        : {CKPT_LOG_FILE}")
print(f"  [CKPT] SLURM_JOB_ID   : {_slurm_job_id}")
print(f"  [CKPT] Signal handlers : SIGTERM, SIGUSR1, SIGUSR2")
print()
print("  Tips SBATCH script:")
print("    #SBATCH --signal=USR1@120  -> SIGUSR1 dikirim 120s sebelum time limit")
print("    #SBATCH --signal=TERM@60   -> SIGTERM dikirim 60s sebelum time limit")

In [ ]:
# Jalankan cell ini setiap kali sebelum memulai/melanjutkan PES loop
_SLURM_TERMINATE = False
_T_JOB_START     = time.time()
print(f"  [CKPT] Flag _SLURM_TERMINATE direset -> False")
print(f"  [CKPT] Timer job dimulai ulang: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  [CKPT] Siap menjalankan PES loop...")

## __PES — POTENTIAL ENERGY SURFACE CH4__ ##

In [ ]:
# Scan panjang ikatan C-H (symmetric stretch, semua 4 H bergerak serempak)
# Ekuilibrium STO-3G: ~1.085 Angstrom
bond_lengths = MOLECULES['CH4']['bond_lengths']
print(f"Jumlah titik PES: {len(bond_lengths)}")
print(f"Range R_CH      : {bond_lengths[0]:.2f} - {bond_lengths[-1]:.2f} Angstrom")
print(f"R values        : {bond_lengths}")

In [ ]:
# ─── Load checkpoint sebelum loop ────────────────────────────────────
results, _done_set = load_checkpoint()
_ckpt_log(f"PES loop started. {len(bond_lengths)-len(_done_set)} points remaining.")

CHEM_ACC = 1.6e-3   # Hartree

print(f"\n{'─'*87}")
print(f"  {'R(A)':>5} | {'E_exact(Ha)':>13} | {'E_VQE(Ha)':>13} | "
      f"{'|dE|(Ha)':>10} | {'Corr%':>6} | Status")
print(f"{'─'*87}")

for R in bond_lengths:
    R_key = round(float(R), 4)

    # ─── Skip titik yang sudah selesai ───────────────────────────────
    if R_key in _done_set:
        r_prev = next((x for x in results if abs(x['bond_length'] - R) < 1e-6), None)
        if r_prev:
            print(f"  {R:5.3f} | {r_prev['E_exact']:13.6f} | {r_prev['E_vqe']:13.6f} | "
                  f"{r_prev['delta_vqe']:10.2e} | {'N/A':>6} | [SKIP]")
        continue

    # ─── Cek SLURM termination signal ────────────────────────────────
    if _SLURM_TERMINATE:
        print(f"\n  [SLURM] Terminate signal aktif. Loop berhenti di R={R:.3f} A.")
        _ckpt_log(f"SLURM terminate at R={R:.3f}. Loop stopped.")
        break

    try:
        t_start = time.time()

        # Build Hamiltonian untuk titik ini
        H_t, T_t, V_t, E_core_t, n_q_t, n_orb_t, n_ae_t, \
            n_spin_orbs_t, n_q_jw_t, U_scbk_t, E_hf_mf_t, H_jw_t = \
            build_hamiltonian_v2(mol_cfg, bond_length=R, use_scbk=USE_SCBK)

        # Energi eksak (FCI)
        evals_t  = np.linalg.eigvalsh(H_t)
        E_exact_t = float(evals_t[0].real)

        # Energi Hartree-Fock
        sc_kw_t  = dict(use_scbk=USE_SCBK, U_scbk=U_scbk_t, n_q_jw=n_q_jw_t)
        psi_hf_t = prepare_hf_state(n_q_t, n_ae_t, H_mat=H_t, **sc_kw_t)
        E_hf_t   = float(expect_value(psi_hf_t, H_t))

        # VQE
        vqe_t = run_vqe_v2(
            H_t, n_q_t, n_ae_t,
            n_orb       = n_orb_t,
            ansatz_type = ANSATZ_TYPE,
            k           = K_LAYERS,
            use_scbk    = USE_SCBK,
            U_scbk      = U_scbk_t,
            n_q_jw      = n_q_jw_t,
            E_core      = E_core_t,
            method      = 'L-BFGS-B',
            n_restarts  = 3,
            seed        = 42,
            verbose     = True,
        )

        E_vqe_t   = float(vqe_t['E_vqe'])
        delta_vqe = abs(E_vqe_t - E_exact_t)
        corr_pct  = ((E_hf_t - E_vqe_t) / (E_hf_t - E_exact_t) * 100
                     if abs(E_hf_t - E_exact_t) > 1e-10 else 0.0)
        t_elapsed = time.time() - t_start

        results.append({
            'bond_length' : float(R),
            'E_exact'     : E_exact_t,
            'E_vqe'       : E_vqe_t,
            'delta_vqe'   : delta_vqe,
            'E_hf'        : E_hf_t,
            'corr_pct'    : corr_pct,
            'n_iters_vqe' : vqe_t['n_iters'],
            'history_vqe' : vqe_t['energy_history'],
            't_elapsed_s' : t_elapsed,
        })
        _done_set.add(R_key)

        # ─── Simpan checkpoint setelah setiap titik ───────────────────
        save_checkpoint(results, _done_set)
        _ckpt_log(f"R={R:.3f}: E_exact={E_exact_t:.6f}, E_VQE={E_vqe_t:.6f}, "
                  f"dE={delta_vqe:.2e}, corr={corr_pct:.1f}%, t={t_elapsed:.1f}s")

        acc_sym = 'OK' if delta_vqe < CHEM_ACC else '--'
        print(f"  {R:5.3f} | {E_exact_t:13.6f} | {E_vqe_t:13.6f} | "
              f"{delta_vqe:10.2e} | {corr_pct:5.1f}% | {acc_sym} ({t_elapsed:.0f}s)")

        # Cek signal setelah setiap titik selesai
        if _SLURM_TERMINATE:
            print(f"\n  [SLURM] Checkpoint tersimpan. Loop keluar dengan aman.")
            break

    except MemoryError:
        msg = f"R={R:.3f}: MemoryError - Hilbert space terlalu besar?"
        print(f"  {R:5.3f} | MEMORY ERROR - {msg}")
        _ckpt_log(msg)
        save_checkpoint(results, _done_set)
        break   # Memory error fatal - stop loop

    except Exception as e:
        msg = f"R={R:.3f}: {type(e).__name__}: {str(e)[:60]}"
        print(f"  {R:5.3f} | ERROR: {type(e).__name__}: {str(e)[:40]}")
        _ckpt_log(msg)
        save_checkpoint(results, _done_set)
        continue   # Skip titik ini, lanjut ke berikutnya

print(f"{'─'*87}")
n_done  = len(_done_set)
n_total = len(bond_lengths)
if n_done == n_total:
    print(f"  PES loop selesai! ({n_done}/{n_total} titik)")
elif _SLURM_TERMINATE:
    print(f"  [SLURM] Job dihentikan scheduler: {n_done}/{n_total} titik selesai.")
    print(f"  Re-submit job dan jalankan ulang cell ini untuk melanjutkan.")
else:
    print(f"  PES loop: {n_done}/{n_total} titik selesai.")

In [ ]:
df = pd.DataFrame([{
    'R_CH (A)'      : f"{r['bond_length']:.3f}",
    'E_exact (Ha)'  : f"{r['E_exact']:.6f}",
    'E_VQE (Ha)'    : f"{r['E_vqe']:.6f}",
    '|dE_VQE| (Ha)' : f"{r['delta_vqe']:.2e}",
    'Corr%'         : f"{r.get('corr_pct',0.0):.1f}",
    'Iter VQE'      : r['n_iters_vqe'],
    't (s)'         : f"{r.get('t_elapsed_s', 0):.0f}",
    'Chem.Acc.'     : 'OK' if r['delta_vqe'] < 1.6e-3 else '--',
} for r in results])

print("Tabel Perbandingan Energi PES CH4:")
print(df.to_string(index=False))

In [ ]:
if results:
    dv = np.array([r['delta_vqe'] for r in results])
    tt = np.array([r.get('t_elapsed_s', 0) for r in results])
    print(f"\n{'─'*55}")
    print(f"Statistik Error Energi:")
    print(f"{'─'*55}")
    print(f"{'Rata-rata |dE|':20s} {np.mean(dv):12.2e}")
    print(f"{'Maksimum |dE|':20s} {np.max(dv):12.2e}")
    print(f"{'Minimum |dE|':20s} {np.min(dv):12.2e}")
    if tt.sum() > 0:
        print(f"{'─'*55}")
        print(f"Waktu komputasi:")
        print(f"{'Total':20s} {tt.sum()/3600:10.2f} jam")
        print(f"{'Rata-rata/titik':20s} {np.mean(tt)/60:10.1f} menit")
    print(f"{'─'*55}")

## __PLOT__ ##

In [ ]:
if len(results) < 2:
    print("Terlalu sedikit titik untuk plot. Jalankan lebih banyak PES points.")
else:
    R_arr  = np.array([r['bond_length'] for r in results])
    E_ex   = np.array([r['E_exact']     for r in results])
    E_vq   = np.array([r['E_vqe']       for r in results])
    dv_arr = np.array([r['delta_vqe']   for r in results])

    fig = plt.figure(figsize=(16, 11))
    gs  = gridspec.GridSpec(2, 2, hspace=0.38, wspace=0.35)

    # ── 1: Kurva energi potensial ─────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(R_arr, E_ex, 'k-x',  lw=2.5, ms=8, zorder=5,
             label='Eksak (FCI diagonalisasi)')
    ax1.plot(R_arr, E_vq, 'ro--', lw=2.0, ms=8, zorder=4, label='VQE k-UpCCGSD')

    idx_min = np.argmin(E_ex)
    ax1.axvline(R_arr[idx_min], color='gray', ls='--', alpha=0.4, lw=1.5)
    ax1.scatter([R_arr[idx_min]], [E_ex[idx_min]], s=180, color='black',
                zorder=10, edgecolor='white', lw=2.5)
    ax1.annotate(f'R_eq = {R_arr[idx_min]:.3f} A\nE = {E_ex[idx_min]:.5f} Ha',
                 xy=(R_arr[idx_min], E_ex[idx_min]),
                 xytext=(R_arr[idx_min] + 0.2, E_ex[idx_min] - 0.05),
                 fontsize=10.5, color='black',
                 bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', alpha=0.9),
                 arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    ax1.set_xlabel('Panjang Ikatan C-H, R (Angstrom)', fontsize=13)
    ax1.set_ylabel('Energi (Hartree)', fontsize=13)
    ax1.set_title(f'Potential Energy Surface CH4 — VQE {ANSATZ_TYPE} k={K_LAYERS} vs Eksak (STO-3G)',
                  fontsize=14, weight='bold')
    ax1.legend(fontsize=11, loc='upper right')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(R_arr[0] - 0.05, R_arr[-1] + 0.05)

    # ── 2: Log error vs eksak ─────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.semilogy(R_arr, dv_arr, 'b^-', lw=2, ms=7, label='|E_VQE - E_eksak|')
    ax2.axhline(1.6e-3, color='green', ls='--', lw=2, alpha=0.85,
                label='Chemical accuracy (1.6 mHa)')
    ax2.fill_between(R_arr, 0, 1.6e-3, alpha=0.08, color='green')
    ax2.set_xlabel('Panjang Ikatan C-H, R (Angstrom)', fontsize=12)
    ax2.set_ylabel('|dE| (Ha)', fontsize=12)
    ax2.set_title('Error Energi VQE vs Eksak — CH4', fontsize=12, weight='bold')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3, which='both')

    # ── 3: Jumlah iterasi VQE ─────────────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.bar(R_arr, [r['n_iters_vqe'] for r in results],
            width=0.06, color='steelblue', alpha=0.75, edgecolor='navy')
    ax3.set_xlabel('Panjang Ikatan C-H, R (Angstrom)', fontsize=12)
    ax3.set_ylabel('Jumlah Iterasi Optimizer', fontsize=12)
    ax3.set_title('Iterasi VQE per Titik PES — CH4', fontsize=12, weight='bold')
    ax3.grid(True, alpha=0.3, axis='y')
    avg_iters = np.mean([r['n_iters_vqe'] for r in results])
    ax3.axhline(avg_iters, color='red', ls='--', lw=1.5, alpha=0.7,
                label=f'Rata-rata: {avg_iters:.0f}')
    ax3.legend(fontsize=10)

    plt.suptitle(f'CH4 CAS({MOLECULES["CH4"]["n_electrons"]-2*MOLECULES["CH4"]["n_frozen_core"]}e,'
                 f'{MOLECULES["CH4"]["n_active_orbs"]}o) | {ANSATZ_TYPE} k={K_LAYERS} | '
                 f'Encoding: {"SCBK" if USE_SCBK else "JW"} | STO-3G',
                 fontsize=11, y=1.01, style='italic')
    plt.savefig('pes_ch4_vqe.pdf', bbox_inches='tight', dpi=150)
    plt.show()